# 03 Fault prediction, evaluation, and geothermal candidates

This notebook trains fault models on the feature stacks from notebooks 01 and 02. It evaluates them with spatially blocked cross-validation. Then it extracts unmapped fault candidates and ranks them by structural settings that favor geothermal systems.

**Method summary.**

1. Engineer multi-scale neighborhood features.
2. Assign 20 km blocks to folds. Exclude a buffer around each test fold from training, so that fault traces that cross block edges do not leak.
3. Train LightGBM as a positive-unlabeled problem. Mapped faults are positives. Cells far from mapped faults are unlabeled and get a low weight. Cells in the band near mapped faults are left out of training, because the mapped trace position is uncertain there.
4. Score with a distance-weighted Tversky index (DTI). This index forgives predictions within 300 m of a mapped trace and penalizes missed faults more than false alarms (alpha = 0.2, beta = 0.8).
5. Optional: train a U-Net on a GPU and compare it with LightGBM on one held-out fold.
6. Extract candidate faults that are far from mapped faults. Rank them by prediction confidence, structural complexity (junctions and terminations), extensional strain, and shallow seismicity.

**Important limit.** The labels are incomplete. A high-confidence prediction far from a mapped fault counts as a false positive in the scores, but it can be a real, unmapped fault. The scores measure agreement with the current map. The candidate list in section 7 is the scientific output. Experts must review it.

**Annotations.** `NOTE:` marks a design decision or a heuristic that you can change.

**Environment.** `lightgbm`, `scikit-image`, `psutil`. Section 6 also needs `torch` and `segmentation-models-pytorch` and a GPU profile.

#### What the next cell does

It loads the libraries that the notebook uses. Nothing is computed yet.

- **`lightgbm`** trains gradient-boosted decision trees. The model is a sequence of small decision trees, each correcting the errors of the ones before it. It handles missing values (NaN) directly, which matters because lidar doesn't cover the whole area.
- **`psutil`** reports how much memory is free, so the notebook can stop before building a feature table that wouldn't fit.
- **`xarray`, `rioxarray`, and `fsspec`** read the zarr files from notebooks 01 and 02, and write the output GeoTIFFs.
- **`scipy.ndimage`** provides the image filters: smoothing, dilation, labeling of connected regions, and distance maps.
- **`skimage.morphology.skeletonize`** thins each predicted fault region to a one-cell-wide centerline.
- **`rasterio.features` and `shapely`** convert the raster candidates into polygons.
- **`geopandas`, `pandas`, `numpy`, and `matplotlib`** handle tables, arrays, and plots.

In [ ]:
import json
import os
import pathlib
import warnings

import fsspec
import geopandas as gpd
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import rasterio.features
import rioxarray  # noqa: F401
import xarray as xr
from rasterio.transform import from_origin
from scipy import ndimage
from shapely.geometry import shape
from skimage.morphology import skeletonize

## 0. Configuration

#### What the next cell does

It only sets values. Edit this cell to change how the notebook runs.

- **Inputs.**
  - `REQUIRE_DEM`: stops if the lidar layers from notebook 02 are missing. Set it to `False` to run without them.
  - `EXCLUDE_FEATURES`: layers to leave out.
  - `RANKING_ONLY_PREFIXES`: layers whose names start with these prefixes (by default `mt_`, the magnetotelluric layers) are never used as model features. They are used only in the candidate ranking. The MT models resolve features tens of kilometres across, so as features they would mostly tell the model where a cell is, not whether it's a fault.
- **Spatial cross-validation.**
  - `SEED` makes the random choices repeatable.
  - The grid is divided into 20 km squares (`BLOCK_PX = 200` cells), which are dealt into `N_FOLDS = 5` groups (folds).
  - `BUFFER_PX = 10` cells (1 km) around each test group is kept out of training.
- **Positive-unlabeled sampling.** The fault map is incomplete, so a cell without a mapped fault isn't known to be fault-free.
  - Cells more than `NEG_MIN_DIST_M` (1 km) from a mapped fault are treated as **unlabeled**, meaning probably not a fault.
  - The notebook uses `UNLABELED_PER_POSITIVE` (5) of them per fault cell.
  - Each unlabeled cell counts half as much (`UNLABELED_WEIGHT`) as a fault cell.
- **Features.**
  - `FEATURE_SCALES_PX = (3, 10)` smooths each layer over 3 cells (300 m) and 10 cells (1 km).
  - `MEMORY_FRACTION_LIMIT` stops the notebook if the features would need more than 60% of the free memory.
- **`LGB_PARAMS`**, the LightGBM settings:
  - `learning_rate`: how much each new tree changes the model. Small steps are slower but more reliable.
  - `num_leaves`: how detailed each tree can be.
  - `min_data_in_leaf`: stops the model from making rules based on fewer than 200 cells.
  - `feature_fraction` and `bagging_fraction`: each tree sees a random 80% of the features and of the cells, which reduces overfitting (learning noise that doesn't generalize).
  - `lambda_l2`: a penalty that also reduces overfitting.
- **Tree count.** `NUM_BOOST_ROUND` is the maximum number of trees. `EARLY_STOP` stops training once 100 trees in a row haven't improved the validation score.
- **Metric.** `DTI_R_M`, `DTI_ALPHA`, and `DTI_BETA` are the settings of the GEMS score (section 1).
- **Post-processing.** `PP_DILATE_PX`, `PP_GAMMA`, and `PP_FLOOR` are the values tried in the post-processing search.
- **Candidates.**
  - `CAND_THRESHOLD`: the minimum probability for a predicted fault.
  - `CAND_MIN_DIST_M`: how far a candidate must be from a mapped fault.
  - `CAND_MIN_LENGTH_M`: the shortest candidate kept.
  - `COMPLEXITY_RADIUS_M`: the neighborhood in which fault junctions and ends are counted.
  - `FAVOR_WEIGHTS`: how the ranking score combines its factors. For every factor, a higher value counts as more favorable. The MT factor is log10 conductivity, so conductive zones rank high.
- **`RUN_UNET`** turns on the optional neural network in section 6. It needs a GPU.

In [ ]:
# Default: a "gems" folder next to the notebooks ($PWD/gems; a Jupyter kernel runs in the notebook's folder).
STORE_ROOT = os.environ.get("GEMS_STORE", str(pathlib.Path.cwd() / "gems"))
CACHE_DIR = pathlib.Path(os.environ.get("GEMS_CACHE", "/tmp/gems_cache"))
REQUIRE_DEM = True             # NOTE: False lets the notebook run without notebook 02 output
DEM_NAME = "dem1m_features.zarr"
EXCLUDE_FEATURES = []          # base layer names to leave out
# Layers used only to rank candidates (section 7), never as model features. NOTE: the MT models resolve
# features tens of km across; as features they would mostly tell the model where a cell is.
RANKING_ONLY_PREFIXES = ("mt_",)

SEED = 42
N_FOLDS = 5
BLOCK_PX = 200                 # 20 km blocks at 100 m
BUFFER_PX = 10                 # 1 km exclusion buffer around each test fold

# Positive-unlabeled sampling. NOTE: heuristic values; test their effect.
NEG_MIN_DIST_M = 1000.0        # unlabeled cells must be at least this far from a mapped fault
UNLABELED_PER_POSITIVE = 5.0
UNLABELED_WEIGHT = 0.5

# Multi-scale features: smoothing scales in cells.
FEATURE_SCALES_PX = (3, 10)
MEMORY_FRACTION_LIMIT = 0.6    # stop if the feature stack needs more than this fraction of free RAM

LGB_PARAMS = {"objective": "binary", "learning_rate": 0.05, "num_leaves": 63,
              "min_data_in_leaf": 200, "feature_fraction": 0.8, "bagging_fraction": 0.8,
              "bagging_freq": 1, "lambda_l2": 1.0, "verbose": -1, "seed": SEED}
NUM_BOOST_ROUND, EARLY_STOP = 3000, 100

# Metric (same form as the GEMS challenge).
DTI_R_M, DTI_ALPHA, DTI_BETA = 300.0, 0.2, 0.8

# Post-processing grid searched on out-of-fold predictions.
PP_DILATE_PX = (0, 1)
PP_GAMMA = (0.5, 1.0, 2.0, 3.0)
PP_FLOOR = (0.0, 0.1, 0.2, 0.3)

# Candidates.
CAND_THRESHOLD = 0.5
CAND_MIN_DIST_M = 500.0        # farther than this from any mapped fault
CAND_MIN_LENGTH_M = 2000.0
COMPLEXITY_RADIUS_M = 2000.0
# Higher value = more favorable for every factor. mt_log10sigma_* is log10 conductivity, so conductors rank high.
FAVOR_WEIGHTS = {"p_mean": 0.35, "complexity": 0.25, "gnss_dilatation": 0.2, "eq_logdens_shallow": 0.1,
                 "mt_log10sigma_10km": 0.1}

RUN_UNET = False               # NOTE: needs a GPU profile

#### What the next cell does

It opens the outputs of notebooks 01 and 02, and extracts the arrays the model needs.

1. It creates the cache folder, and a `predictions` folder in the store.
2. It opens `features.zarr` and `labels.zarr`. Opening doesn't read the data yet.
3. It reads the grid definition from the attributes, and rebuilds `TRANSFORM`, the rule that converts rows and columns to map coordinates.
4. **Lidar layers.**
   - If `dem1m_features.zarr` exists, it adds those layers.
   - If it doesn't, the cell stops, or with `REQUIRE_DEM = False`, it warns and continues without them.
5. **Merge.** `xr.merge(..., join="exact")` stops unless every file uses exactly the same grid. This guarantees that the layers line up cell for cell.
6. **Arrays.**
   - `valid`: the survey footprint.
   - `fault`: the mapped fault cells.
   - `fault_dist`: the distance from each cell to the nearest mapped fault.
   - `base_names`: every input layer, except the mask, the layers you excluded, and the ranking-only layers (`mt_…`). The cell prints the ranking-only layers it found.
7. It prints the number of layers, the valid area, and the number of fault cells in it.

In [ ]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
fs, _ = fsspec.core.url_to_fs(STORE_ROOT)
def store_path(rel):
    return f"{STORE_ROOT.rstrip('/')}/{rel}"
fs.makedirs(store_path("predictions"), exist_ok=True)

feat = xr.open_zarr(store_path("features.zarr"))
lab = xr.open_zarr(store_path("labels.zarr"))
GRID = json.loads(feat.attrs["grid"])
H, W, RES = GRID["height"], GRID["width"], GRID["res"]
TRANSFORM = from_origin(GRID["xmin"], GRID["ymax"], RES, RES)

parts = [feat]
if fs.exists(store_path(DEM_NAME)):
    parts.append(xr.open_zarr(store_path(DEM_NAME)))
    print("Loaded", DEM_NAME)
elif REQUIRE_DEM:
    raise RuntimeError(f"{DEM_NAME} not found. Run notebook 02, or set REQUIRE_DEM = False.")
else:
    warnings.warn(f"{DEM_NAME} not found. Running without lidar features.")
ds = xr.merge(parts, join="exact", compat="override")

valid = ds["valid"].values.astype(bool)
fault = lab["fault"].values.astype(bool)
fault_dist = lab["fault_dist_m"].values
NON_FEATURES = {"valid", "spatial_ref"}
ranking_only = [n for n in ds.data_vars if n.startswith(RANKING_ONLY_PREFIXES)]
base_names = [n for n in ds.data_vars
              if n not in NON_FEATURES and n not in EXCLUDE_FEATURES and n not in ranking_only]
if ranking_only:
    print("Ranking only, not model features:", ranking_only)
print(f"{len(base_names)} base layers; valid area {valid.mean():.1%}; fault cells in valid area {int((fault & valid).sum()):,}")

## 1. Distance-weighted Tversky index

This is a local implementation of the metric from the GEMS problem description. `M(g)` is the best kernel-weighted prediction within `R` of ground-truth cell `g`. `TP = sum M(g)`, `FN = sum (1 - M(g))`, and `FP = sum p(x) (1 - k(d(x)))`, where `d(x)` is the distance from `x` to the nearest ground-truth cell and `k(d) = max(1 - d / R, 0)`.

#### What the next cell does

It defines the score used to judge a prediction map: the distance-weighted Tversky index (DTI) from the GEMS challenge. It compares a map of fault probabilities with the map of mapped faults, and it gives credit for near misses, since mapped fault traces are not exact.

**`_shift(a, dy, dx)`** moves a whole map by `dy` rows and `dx` columns, filling the uncovered edge with zeros.

**`dti(p, g, mask, ...)`**:

1. **Mask.** Outside `mask`, the prediction is set to 0 and fault cells are ignored. A single fold can then be scored on its own.
2. **Best nearby prediction (`M`).** For each cell, it finds the highest prediction within `R` (300 m), discounted by distance. The discount weight is 1 at the cell itself and falls linearly to 0 at 300 m. The code does this by shifting the whole prediction map in every direction within the radius and keeping the maximum.
3. **Credit and misses.**
   - **TP (true positive)** adds up `M` over the fault cells: how well each mapped fault cell was found.
   - **FN (false negative)** adds up `1 − M` over the fault cells: how much was missed.
4. **False alarms (FP).** Each predicted cell costs its probability times `(1 − closeness)`, where closeness is 1 on a mapped fault and 0 at 300 m or farther. A prediction right on a fault costs nothing, and one far from any fault costs its full probability.
5. **Score.** `TP / (TP + 0.2·FP + 0.8·FN)`. A score of 1 is perfect. With these weights, missing a fault costs four times as much as a false alarm, because unmapped faults are expected.

The function returns the score and its three parts.

In [ ]:
def _shift(a, dy, dx):
    """out[y, x] = a[y + dy, x + dx], zero outside."""
    out = np.zeros_like(a)
    h, w = a.shape
    ys, yd = (slice(dy, h), slice(0, h - dy)) if dy >= 0 else (slice(0, h + dy), slice(-dy, h))
    xs, xd = (slice(dx, w), slice(0, w - dx)) if dx >= 0 else (slice(0, w + dx), slice(-dx, w))
    out[yd, xd] = a[ys, xs]
    return out

def dti(p, g, mask=None, res=100.0, R=300.0, alpha=0.2, beta=0.8, eps=1e-9):
    p = np.nan_to_num(np.asarray(p, np.float64))
    g = np.asarray(g, bool)
    if mask is not None:
        p = np.where(mask, p, 0.0)
        g = g & mask
    r = int(np.floor(R / res))
    M = np.zeros_like(p)
    for dy in range(-r, r + 1):
        for dx in range(-r, r + 1):
            k = 1.0 - res * np.hypot(dy, dx) / R
            if k > 0:
                np.maximum(M, _shift(p, dy, dx) * k, out=M)
    tp = float(M[g].sum())
    fn = float((1.0 - M[g]).sum())
    if g.any():
        kd = np.clip(1.0 - ndimage.distance_transform_edt(~g) * res / R, 0.0, None)
    else:
        kd = np.zeros_like(p)
    fp = float((p * (1.0 - kd))[p > 0].sum())
    return tp / (tp + alpha * fp + beta * fn + eps), {"TP": tp, "FP": fp, "FN": fn}

## 2. Features

For every base layer the notebook adds NaN-aware Gaussian smooths at each scale in `FEATURE_SCALES_PX`, the residual (layer minus smooth), and the gradient magnitude of the finest smooth. The orientation layers (`*_sin2`, `*_cos2`) and coverage layers only get the finest smooth.

#### What the next cell does

It defines the functions that build extra features from each input layer. It only defines them.

- **`nan_gaussian(a, sigma)`** smooths a map that has gaps, as in notebook 02. It averages only the valid neighbors, and returns NaN where almost none exist.
- **`engineer(name, a, scales)`** returns several versions of one layer:
  - **The original layer.**
  - **A smoothed version at each scale** (300 m and 1 km). This gives the model context: what the values are like around the cell.
  - **The residual at each scale**: the original minus the smoothed version. It shows whether the cell stands out from its surroundings, like a local high or low.
  - **The gradient of the finest smoothed version**: how quickly the layer changes across the cell. This marks edges.
  - **Orientation and coverage layers** (`*_sin2`, `*_cos2`, `*cov`) get only the finest smoothed version. A residual or gradient of an angle component or of coverage doesn't mean anything useful.

In [ ]:
def nan_gaussian(a, sigma):
    m = np.isfinite(a)
    num = ndimage.gaussian_filter(np.where(m, a, 0).astype(np.float32), sigma)
    den = ndimage.gaussian_filter(m.astype(np.float32), sigma)
    with np.errstate(invalid="ignore", divide="ignore"):
        out = num / den
    out[den < 1e-3] = np.nan
    return out.astype(np.float32)

def engineer(name, a, scales):
    out = {name: a}
    light = name.endswith(("_sin2", "_cos2")) or name.endswith("cov")
    for i, s in enumerate(scales):
        sm = nan_gaussian(a, s)
        out[f"{name}__g{s}"] = sm
        if light:
            break
        out[f"{name}__r{s}"] = (a - sm).astype(np.float32)
        if i == 0:
            gr, gc = np.gradient(sm)
            out[f"{name}__grad{s}"] = np.hypot(gr, gc).astype(np.float32)
    return out

#### What the next cell does

It builds all the features and a way to turn them into a table for the model.

1. **Memory check.** It estimates the number of features and the memory they need: 4 bytes per cell per feature. It compares this with 60% of the free memory, and stops with a `MemoryError` before the kernel crashes. If that happens, reduce `FEATURE_SCALES_PX`, add layers to `EXCLUDE_FEATURES`, or use a larger server.
2. **Build.** It runs `engineer` on every base layer and stores all the resulting maps in `F`. `feature_names` keeps their order.
3. **`gather(rows, cols)`** picks the values of every feature at the given cells, and returns a table with one row per cell and one column per feature. This is how the training examples and prediction inputs are built.

In [ ]:
n_est = sum(1 + (1 if n.endswith(("_sin2", "_cos2", "cov")) else 2 * len(FEATURE_SCALES_PX) + 1) for n in base_names)
need = n_est * H * W * 4
avail = psutil.virtual_memory().available
print(f"About {n_est} features, {need / 1e9:.1f} GB; {avail / 1e9:.1f} GB available")
if need > MEMORY_FRACTION_LIMIT * avail:
    raise MemoryError("Feature stack is too large. Use fewer scales, EXCLUDE_FEATURES, or a larger server.")

F = {}
for n in base_names:
    F.update(engineer(n, ds[n].values.astype(np.float32), FEATURE_SCALES_PX))
feature_names = list(F)
print(len(feature_names), "features")

def gather(rows, cols):
    return np.column_stack([F[n][rows, cols] for n in feature_names])

## 3. Spatial folds and positive-unlabeled sampling

Blocks are dealt to folds in order of their fault-cell count, so each fold gets a similar share of faults.

#### What the next cell does

It sets up spatial cross-validation and defines the training functions. Cross-validation estimates how well the model predicts places it hasn't seen.

**Why blocks.** Neighboring cells are nearly identical. If training and test cells were picked at random, the model would be tested on cells right next to cells it trained on, and the score would look better than it really is. So the grid is split into 20 km squares, and whole squares go to one fold.

1. **Blocks.** `block_id` gives every cell the number of its 20 km square.
2. **Balanced folds.** It counts the fault cells in each square. It sorts the squares by that count, with a little random noise to break ties, and deals them to the 5 folds in turn, like dealing cards. Each fold then gets a similar share of faults. `fold` is the resulting map of fold numbers.

**`sample_training(train_mask, rng)`** chooses training examples inside the training area:
- **Positives:** every mapped fault cell, with label 1 and weight 1.
- **Unlabeled:** a random choice of cells more than 1 km from any mapped fault. It takes up to 5 per positive, with label 0 and weight 0.5. The lower weight reflects that some of these cells may be unmapped faults.
- Cells within 1 km of a fault, but not on it, are not used at all. The mapped trace may be slightly off, so their labels are uncertain.

**`fit_lgb(...)`** trains a LightGBM model:
- **Without a tree count**, it holds back 15% of the training squares, again whole squares, to watch the validation score. It stops adding trees when the score stops improving (early stopping).
- **With `num_round`**, it trains exactly that many trees on all the examples.

**`predict_mask(model, mask)`** predicts a fault probability for every cell in `mask`. It works in batches of 2 million cells to limit memory, and returns a full-size map.

In [ ]:
rng = np.random.default_rng(SEED)
nby, nbx = int(np.ceil(H / BLOCK_PX)), int(np.ceil(W / BLOCK_PX))
block_id = (np.arange(H)[:, None] // BLOCK_PX) * nbx + (np.arange(W)[None, :] // BLOCK_PX)
counts = np.bincount(block_id[fault & valid], minlength=nby * nbx).astype(float)
order = np.argsort(counts + rng.random(counts.size) * 0.5)[::-1]
fold_of_block = np.empty(nby * nbx, int)
fold_of_block[order] = np.arange(order.size) % N_FOLDS
fold = fold_of_block[block_id]

def sample_training(train_mask, rng):
    pr, pc = np.nonzero(train_mask & fault)
    ur, uc = np.nonzero(train_mask & (fault_dist > NEG_MIN_DIST_M))
    n_u = min(len(ur), int(len(pr) * UNLABELED_PER_POSITIVE))
    pick = rng.choice(len(ur), n_u, replace=False)
    rows = np.concatenate([pr, ur[pick]])
    cols = np.concatenate([pc, uc[pick]])
    y = np.concatenate([np.ones(len(pr)), np.zeros(n_u)]).astype(np.float32)
    w = np.concatenate([np.ones(len(pr)), np.full(n_u, UNLABELED_WEIGHT)]).astype(np.float32)
    return rows, cols, y, w

def fit_lgb(rows, cols, y, w, rng, num_round=None):
    X = gather(rows, cols)
    if num_round is None:
        # Early stopping on 15 % of the training blocks.
        blocks = np.unique(block_id[rows, cols])
        es_blocks = rng.choice(blocks, max(1, int(0.15 * blocks.size)), replace=False)
        es = np.isin(block_id[rows, cols], es_blocks)
        dtr = lgb.Dataset(X[~es], y[~es], weight=w[~es], feature_name=feature_names, free_raw_data=True)
        dva = lgb.Dataset(X[es], y[es], weight=w[es], reference=dtr)
        return lgb.train(LGB_PARAMS, dtr, NUM_BOOST_ROUND, valid_sets=[dva],
                         callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(0)])
    return lgb.train(LGB_PARAMS, lgb.Dataset(X, y, weight=w, feature_name=feature_names), num_round)

def predict_mask(model, mask, chunk=2_000_000):
    rows, cols = np.nonzero(mask)
    out = np.full((H, W), np.nan, np.float32)
    for i in range(0, rows.size, chunk):
        r, c = rows[i:i + chunk], cols[i:i + chunk]
        out[r, c] = model.predict(gather(r, c), num_iteration=model.best_iteration or None)
    return out

## 4. Cross-validated LightGBM

#### What the next cell does

It runs the cross-validation. For each of the 5 folds:

1. **Test area:** the cells of this fold inside the survey.
2. **Buffer:** it grows the test squares by 1 km. The training area is everything valid outside that. The gap keeps a fault line that crosses a square's edge from appearing in both the training and the test data.
3. **Train:** it samples the training examples and trains a model with early stopping, and records how many trees the model needed.
4. **Predict:** it predicts the test cells and writes them into `oof`, the "out-of-fold" map. When all 5 folds are done, every valid cell has a prediction from a model that never saw that cell. This gives an honest map for evaluation.
5. **Score:** it computes the DTI on this fold, and prints the number of examples, the number of trees, the score, and its parts.

At the end, it prints the mean and standard deviation of the 5 scores. A large spread means the model works well in some regions and poorly in others.

In [ ]:
oof = np.full((H, W), np.nan, np.float32)
fold_scores, best_iters = [], []
for k in range(N_FOLDS):
    test = (fold == k) & valid
    near_test = ndimage.binary_dilation(fold == k, iterations=BUFFER_PX)
    train = valid & ~near_test
    rows, cols, y, w = sample_training(train, rng)
    model = fit_lgb(rows, cols, y, w, rng)
    best_iters.append(model.best_iteration)
    oof[test] = predict_mask(model, test)[test]
    score, parts_ = dti(oof, fault, mask=test, res=RES, R=DTI_R_M, alpha=DTI_ALPHA, beta=DTI_BETA)
    fold_scores.append(score)
    print(f"fold {k}: {int(y.sum()):,} pos, {int((y == 0).sum()):,} unlabeled, "
          f"{model.best_iteration} trees, DTI {score:.3f}  {parts_}")
print(f"Mean DTI (raw probabilities): {np.mean(fold_scores):.3f} +/- {np.std(fold_scores):.3f}")

#### What the next cell does

It tests simple adjustments to the probability map that can raise the DTI score, and picks the best one.

**`postprocess(p, dilate, gamma, floor)`** applies three optional adjustments:
- **`dilate`** widens predicted lines by 1 cell with a maximum filter. This helps when predictions sit just beside the mapped trace.
- **`gamma`** raises every probability to a power. Values above 1 push weak predictions toward 0 and keep strong ones. Values below 1 do the opposite.
- **`floor`** sets probabilities below a threshold to 0. This removes low-level haze, which the score would count as false alarms.

**The search** tries all 32 combinations (2 × 4 × 4). For each one, it scores the out-of-fold map on every fold and averages the scores. It shows the best 10 and stores the best settings in `BEST_PP`.

The settings are chosen on the same map they are scored on, so the best score here is slightly optimistic. Compare it with the unadjusted score from the previous cell.

In [ ]:
def postprocess(p, dilate, gamma, floor):
    q = np.nan_to_num(p)
    if dilate:
        q = ndimage.maximum_filter(q, size=2 * dilate + 1)
    q = q ** gamma
    q[q < floor] = 0.0
    return np.where(valid, q, np.nan).astype(np.float32)

# NOTE: the settings are tuned on the same out-of-fold map that is scored, so the tuned
# score is slightly optimistic. Compare it with the raw score above.
grid_results = []
for d in PP_DILATE_PX:
    for g_ in PP_GAMMA:
        for fl in PP_FLOOR:
            s = np.mean([dti(postprocess(oof, d, g_, fl), fault, mask=(fold == k) & valid, res=RES,
                             R=DTI_R_M, alpha=DTI_ALPHA, beta=DTI_BETA)[0] for k in range(N_FOLDS)])
            grid_results.append({"dilate": d, "gamma": g_, "floor": fl, "dti": s})
pp = pd.DataFrame(grid_results).sort_values("dti", ascending=False)
display(pp.head(10))
BEST_PP = pp.iloc[0][["dilate", "gamma", "floor"]].to_dict()
BEST_PP["dilate"] = int(BEST_PP["dilate"])
print("Best post-processing:", BEST_PP)

## 5. Final model, feature contributions, and prediction map

The final model trains on all folds with the median number of trees from cross-validation. LightGBM computes SHAP contributions natively with `pred_contrib=True`.

#### What the next cell does

It trains the final model, shows which features it relies on, and writes the prediction maps.

1. **Final model.** It samples training examples from the whole survey area and trains a model. It uses the median number of trees found in cross-validation, so no data is held back for early stopping.
2. **Feature contributions (SHAP).** For a random 50,000 training cells, LightGBM computes how much each feature pushed that cell's prediction up or down (`pred_contrib=True`). The average of the absolute values measures how much the model relies on each feature. The bar chart shows the top 25.
   - Check that the model relies on geological signals.
   - A model that mostly uses coverage layers or survey edges is picking up how the data were collected, not the geology.
3. **Prediction.** It predicts every valid cell (`p_raw`), and applies the best post-processing (`p_final`).
4. **`write_cog(arr, rel)`** saves a map as a Cloud Optimized GeoTIFF: a GeoTIFF arranged so that GIS software can read parts of it efficiently over the internet. It records the coordinate system, the position, and NaN as "no data". It writes the file to the cache and then copies it to the store.
5. It writes three maps to `predictions/`:
   - `fault_prob_raw.tif`: the final model's raw probabilities.
   - `fault_prob_post.tif`: after post-processing. Use this one for candidates.
   - `fault_prob_oof.tif`: the out-of-fold map, for honest evaluation.

In [ ]:
rows, cols, y, w = sample_training(valid, rng)
final = fit_lgb(rows, cols, y, w, rng, num_round=int(np.median(best_iters)))

sub = rng.choice(rows.size, min(50_000, rows.size), replace=False)
contrib = final.predict(gather(rows[sub], cols[sub]), pred_contrib=True)[:, :-1]
imp = pd.Series(np.abs(contrib).mean(axis=0), index=feature_names).sort_values(ascending=False)
imp.head(25)[::-1].plot.barh(figsize=(7, 8), title="Mean |SHAP| (top 25)")
plt.tight_layout()

p_raw = predict_mask(final, valid)
p_final = postprocess(p_raw, **BEST_PP)

def write_cog(arr, rel):
    da = xr.DataArray(arr, coords={"y": ds.y, "x": ds.x}, dims=("y", "x")).rio.write_crs(GRID["crs"])
    da = da.rio.write_transform(TRANSFORM).rio.write_nodata(np.nan)
    local = CACHE_DIR / pathlib.Path(rel).name
    da.rio.to_raster(local, driver="COG", dtype="float32")
    fs.put(str(local), store_path(rel))

write_cog(p_raw, "predictions/fault_prob_raw.tif")
write_cog(p_final, "predictions/fault_prob_post.tif")
write_cog(np.where(valid, oof, np.nan).astype(np.float32), "predictions/fault_prob_oof.tif")

#### What the next cell does

It shows the post-processed fault probability map, from dark (0) to bright (1), with the mapped Quaternary faults drawn as thin cyan lines.

Bright lines that follow the cyan lines are faults the model found again. Bright linear features far from any cyan line are possible unmapped faults, and section 7 extracts them.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(p_final, cmap="magma", vmin=0, vmax=1)
ax.contour(fault, levels=[0.5], colors="cyan", linewidths=0.3)
ax.set_title("Fault probability (post-processed); cyan = mapped Quaternary faults")
ax.set_axis_off()

## 6. Optional: U-Net on a GPU

This section trains a U-Net on the base layers of the training folds and compares it with LightGBM on fold `HOLDOUT_FOLD`. The loss is a masked Tversky loss with the metric weights plus binary cross-entropy. The loss ignores the uncertain band near mapped faults, as the LightGBM sampling does.

Set `RUN_UNET = True` in section 0 to run it. The cell stops if no GPU is available.

#### What the next cell does

It defines, and when `RUN_UNET = True` runs, an optional U-Net: a convolutional neural network that looks at whole image patches instead of one cell at a time. It can learn shapes, such as long thin lines, that a cell-by-cell model can't see. The U-Net is then compared with LightGBM on one held-out fold.

**Settings:**
- `HOLDOUT_FOLD`: the fold kept for testing.
- `PATCH`: 128 × 128 cells (12.8 km) per training patch.
- `BATCH`: 16 patches per training step.
- `STEPS`: 400 steps per epoch (one pass of training).
- `EPOCHS`: 15.

**`run_unet()`:**
1. **GPU check.** It stops if no GPU is available.
2. **Training area.** This is everything valid except the held-out fold and a 1 km buffer around it.
3. **Input channels.** Each base layer is scaled robustly: its median is subtracted, and it's divided by its interquartile range, both measured on the training area only. The result is clipped to ±10, and gaps are set to 0. The valid mask is added as one more channel, so the network knows where there's no data.
4. **Loss mask.** The training counts only cells in the training area. It also ignores the cells within 1 km of a mapped fault that aren't fault cells themselves. That's the same uncertain band that the LightGBM sampling leaves out.
5. **Batches.** Each patch is centred either on a random fault cell or on a random cell of the training area, with equal chance. Faults are rare, so this makes sure the network sees enough of them.
6. **Loss.** The loss is a Tversky loss with the same weights as the score, which rewards found faults and penalizes misses more than false alarms, plus binary cross-entropy for stable training. Both count only the cells in the loss mask.
7. **Training.** The network is a U-Net with a ResNet-34 encoder, starting from random weights. It trains for 15 epochs with the AdamW optimizer and a learning rate that decreases gradually (cosine schedule), and prints the average loss per epoch.
8. **Prediction.** It slides a 256-cell window over the whole grid in steps of 192 cells, so neighboring windows overlap by 64 cells. It averages the overlapping predictions, which avoids visible seams, and sets cells outside the survey to NaN.

**If `RUN_UNET` is `True`,** it prints the DTI on the held-out fold for three maps: LightGBM (its out-of-fold predictions), the U-Net, and the average of the two. It also writes `fault_prob_unet.tif`.

In [ ]:
HOLDOUT_FOLD, PATCH, BATCH, STEPS, EPOCHS = 0, 128, 16, 400, 15

def run_unet():
    import torch
    import torch.nn.functional as TF
    import segmentation_models_pytorch as smp
    if not torch.cuda.is_available():
        raise RuntimeError("RUN_UNET is True, but no GPU is available. Select a GPU profile.")
    dev = torch.device("cuda")

    train_area = valid & ~ndimage.binary_dilation(fold == HOLDOUT_FOLD, iterations=BUFFER_PX)
    chans = []
    for n in base_names:
        a = ds[n].values.astype(np.float32)
        v = a[train_area & np.isfinite(a)]
        med, iqr = (np.median(v), np.subtract(*np.percentile(v, [75, 25])) + 1e-6) if v.size else (0.0, 1.0)
        chans.append(np.clip(np.nan_to_num((a - med) / iqr), -10, 10))
    chans.append(valid.astype(np.float32))
    X = np.stack(chans)
    Y = fault.astype(np.float32)
    uncertain = (fault_dist > 0) & (fault_dist <= NEG_MIN_DIST_M)
    Mtrain = (train_area & ~uncertain).astype(np.float32)

    centers_pos = np.argwhere(train_area & fault)
    centers_any = np.argwhere(train_area)
    half = PATCH // 2
    def batch():
        xs, ys, ms = [], [], []
        for _ in range(BATCH):
            c = centers_pos[rng.integers(len(centers_pos))] if rng.random() < 0.5 else centers_any[rng.integers(len(centers_any))]
            r0 = int(np.clip(c[0] - half, 0, H - PATCH)); c0 = int(np.clip(c[1] - half, 0, W - PATCH))
            xs.append(X[:, r0:r0 + PATCH, c0:c0 + PATCH]); ys.append(Y[r0:r0 + PATCH, c0:c0 + PATCH])
            ms.append(Mtrain[r0:r0 + PATCH, c0:c0 + PATCH])
        t = lambda a: torch.from_numpy(np.stack(a)).to(dev)
        return t(xs), t(ys)[:, None], t(ms)[:, None]

    def loss_fn(logits, y, m, a=DTI_ALPHA, b=DTI_BETA):
        p = torch.sigmoid(logits)
        tp, fp, fn = (p * y * m).sum(), (p * (1 - y) * m).sum(), ((1 - p) * y * m).sum()
        tversky = 1 - (tp + 1) / (tp + a * fp + b * fn + 1)
        bce = (TF.binary_cross_entropy_with_logits(logits, y, reduction="none") * m).sum() / m.sum().clamp(min=1)
        return tversky + bce

    net = smp.Unet("resnet34", encoder_weights=None, in_channels=X.shape[0], classes=1).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * STEPS)
    for ep in range(EPOCHS):
        net.train(); tot = 0.0
        for _ in range(STEPS):
            xb, yb, mb = batch()
            loss = loss_fn(net(xb), yb, mb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tot += loss.item()
        print(f"epoch {ep}: loss {tot / STEPS:.4f}")

    # Sliding-window prediction over the whole grid.
    net.eval(); win, stride = 256, 192
    Hp, Wp = int(np.ceil((H - win) / stride)) * stride + win, int(np.ceil((W - win) / stride)) * stride + win
    Xp = np.pad(X, ((0, 0), (0, max(Hp - H, 0)), (0, max(Wp - W, 0))), mode="reflect")
    acc = np.zeros(Xp.shape[1:], np.float32); cnt = np.zeros_like(acc)
    with torch.no_grad():
        for r0 in range(0, Xp.shape[1] - win + 1, stride):
            for c0 in range(0, Xp.shape[2] - win + 1, stride):
                xb = torch.from_numpy(Xp[None, :, r0:r0 + win, c0:c0 + win]).to(dev)
                acc[r0:r0 + win, c0:c0 + win] += torch.sigmoid(net(xb))[0, 0].cpu().numpy()
                cnt[r0:r0 + win, c0:c0 + win] += 1
    return np.where(valid, (acc / np.maximum(cnt, 1))[:H, :W], np.nan).astype(np.float32)

if RUN_UNET:
    p_unet = run_unet()
    ho = (fold == HOLDOUT_FOLD) & valid
    for name, p in [("LightGBM (OOF)", oof), ("U-Net", p_unet), ("Mean of both", np.nanmean([oof, p_unet], axis=0))]:
        print(f"{name:16s} DTI on fold {HOLDOUT_FOLD}: {dti(p, fault, mask=ho, res=RES, R=DTI_R_M, alpha=DTI_ALPHA, beta=DTI_BETA)[0]:.3f}")
    write_cog(p_unet, "predictions/fault_prob_unet.tif")

## 7. Unmapped fault candidates and geothermal ranking

A candidate is a connected region where the post-processed probability is at least `CAND_THRESHOLD` and the distance to the nearest mapped fault is more than `CAND_MIN_DIST_M`. The notebook skeletonizes each candidate to measure its length and strike.

**Structural complexity.** Geothermal systems in the Great Basin favor step-overs, fault terminations, fault intersections, and accommodation zones. The notebook skeletonizes the combined network of mapped faults and candidates. It counts junctions (three or more neighbors) and endpoints (one neighbor) within `COMPLEXITY_RADIUS_M` of each candidate.

**Score.** The score is a weighted sum of percentile ranks (`FAVOR_WEIGHTS`). `NOTE:` this is a screening heuristic, not a calibrated favorability model. Compare the list with the INGENIOUS favorability maps.

#### What the next cell does

It finds possible unmapped faults (candidates) and measures each one.

1. **Candidate cells.** A cell is a candidate if it meets all three conditions:
   - its post-processed probability is at least `CAND_THRESHOLD` (0.5),
   - it's more than `CAND_MIN_DIST_M` (500 m) from any mapped fault,
   - it's inside the survey.
2. **Regions.** `ndimage.label` gives each connected group of candidate cells its own number. Diagonal neighbors count as connected.
3. **Centerlines.** `skeletonize` thins each region to a one-cell-wide centerline.
4. **Structural complexity.** Geothermal systems in the Great Basin favor places where faults end, meet, or step over.
   - It thins the combined network of mapped faults and candidates, and counts each centerline cell's neighbors.
   - Three or more neighbors mark a **junction**, where lines meet or branch. One neighbor marks an **endpoint**, where a line ends.
   - `complexity_map` counts the junctions and endpoints within a square of about 2 km radius around every cell.
5. **Per candidate**, it measures:
   - **Length:** the number of centerline cells × 100 m. Diagonal steps count as 100 m, so this is approximate. Candidates shorter than `CAND_MIN_LENGTH_M` (2 km) are dropped.
   - **Strike:** the main direction of the centerline, found as the direction in which its points spread most (the principal eigenvector of their covariance). It's given as a compass angle from 0° to 180°.
   - The **mean and maximum probability**, the **maximum complexity**, and the **closest distance to a mapped fault**.
   - The mean over the region of every other factor in `FAVOR_WEIGHTS`, when that layer exists. By default, these are GNSS dilatation, shallow earthquake density, and MT conductivity at 10 km.
6. It prints the number of candidate regions and how many are long enough to keep.

In [ ]:
EIGHT = np.ones((3, 3), int)

cand_bin = (np.nan_to_num(p_final) >= CAND_THRESHOLD) & (fault_dist > CAND_MIN_DIST_M) & valid
cand_lab, n_cand = ndimage.label(cand_bin, structure=EIGHT)
cand_skel = skeletonize(cand_bin)

network = skeletonize(fault | cand_bin)
nbrs = ndimage.convolve(network.astype(int), EIGHT, mode="constant") - network
junction = network & (nbrs >= 3)
endpoint = network & (nbrs == 1)
win = 2 * int(COMPLEXITY_RADIUS_M / RES) + 1
complexity_map = (ndimage.uniform_filter(junction.astype(np.float32), win) +
                  ndimage.uniform_filter(endpoint.astype(np.float32), win)) * win * win

xs = GRID["xmin"] + RES * (np.arange(W) + 0.5)
ys = GRID["ymax"] - RES * (np.arange(H) + 0.5)
recs = []
for i, sl in enumerate(ndimage.find_objects(cand_lab), start=1):
    comp = cand_lab[sl] == i
    sk = cand_skel[sl] & comp
    length = sk.sum() * RES                      # NOTE: approximate; diagonal steps count as RES
    if length < CAND_MIN_LENGTH_M:
        continue
    rr, cc = np.nonzero(sk)
    px, py = xs[cc + sl[1].start], ys[rr + sl[0].start]
    if px.size >= 2:
        evals, evecs = np.linalg.eigh(np.cov(np.vstack([px, py])))
        vx, vy = evecs[:, -1]
        strike = float(np.degrees(np.arctan2(vx, vy)) % 180.0)
    else:
        strike = np.nan
    rec = {"cand_id": i, "length_m": float(length), "strike_deg": strike,
           "p_mean": float(np.nanmean(p_final[sl][comp])), "p_max": float(np.nanmax(p_final[sl][comp])),
           "complexity": float(complexity_map[sl][comp].max()),
           "dist_to_mapped_m": float(fault_dist[sl][comp].min())}
    for extra in [c for c in FAVOR_WEIGHTS if c not in ("p_mean", "complexity")]:
        if extra in ds:
            rec[extra] = float(np.nanmean(ds[extra].values[sl][comp]))
    recs.append(rec)

cands = pd.DataFrame(recs)
print(f"{n_cand} candidate regions, {len(cands)} longer than {CAND_MIN_LENGTH_M / 1000:.0f} km")

#### What the next cell does

It ranks the candidates and saves them as polygons for GIS software.

1. **Ranking score.** Each factor in `FAVOR_WEIGHTS` has different units, so each is first converted to a **percentile rank** among the candidates: 0 for the lowest value and 1 for the highest. The score is the weighted sum of these ranks. By default, it combines probability (35%), complexity (25%), dilatation (20%), shallow seismicity (10%), and MT conductivity at 10 km (10%). A missing factor is left out with a warning, and the remaining weights are rescaled so the score still runs from 0 to 1. The candidates are sorted from highest to lowest score.
2. **Polygons.** `rasterio.features.shapes` traces the outline of each kept candidate region as a polygon. Pieces with the same candidate number are merged (`dissolve`), and the measurements and the score are attached.
3. **Save.** It writes `fault_candidates.gpkg` to the cache, copies it to `predictions/` in the store, and shows the 20 highest-ranked candidates.

This is a screening heuristic, not a calibrated favorability model. Experts need to review the list.

In [ ]:
score = np.zeros(len(cands))
used = 0.0
for col, wgt in FAVOR_WEIGHTS.items():
    if col not in cands or cands[col].isna().all():
        warnings.warn(f"{col} is not available. It is left out of the score.")
        continue
    score += wgt * cands[col].rank(pct=True).fillna(0.0).values
    used += wgt
cands["favorability"] = score / max(used, 1e-9)
cands = cands.sort_values("favorability", ascending=False).reset_index(drop=True)

keep = np.isin(cand_lab, cands.cand_id.values)
polys = [(shape(geom), int(v)) for geom, v in rasterio.features.shapes(
    cand_lab.astype(np.int32), mask=keep, transform=TRANSFORM, connectivity=8)]
gpoly = gpd.GeoDataFrame({"cand_id": [v for _, v in polys]}, geometry=[g for g, _ in polys], crs=GRID["crs"])
gpoly = gpoly.dissolve("cand_id").reset_index().merge(cands, on="cand_id")
gpoly = gpoly.sort_values("favorability", ascending=False)

local = CACHE_DIR / "fault_candidates.gpkg"
gpoly.to_file(local, driver="GPKG")
fs.put(str(local), store_path("predictions/fault_candidates.gpkg"))
display(cands.head(20))

#### What the next cell does

It maps the 25 highest-ranked candidates, in red, on the survey area, in light grey. The mapped Quaternary faults are drawn as dark grey lines.

Use it to see where the top candidates lie relative to the known faults. Then inspect them in `fault_candidates.gpkg`, together with the hillshade and the magnetic layers.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(np.where(valid, 1.0, np.nan), cmap="Greys", vmin=0, vmax=3)
ax.contour(fault, levels=[0.5], colors="0.3", linewidths=0.3)
top = np.isin(cand_lab, cands.cand_id.head(25).values)
ax.imshow(np.where(top, 1.0, np.nan), cmap="autumn", vmin=0, vmax=1)
ax.set_title("Top 25 unmapped fault candidates (red) with mapped Quaternary faults (grey)")
ax.set_axis_off()

## 8. Next steps

* Review the top candidates against the 1 m hillshade, the GeoDAWN magnetic derivatives, and published geologic maps.
* Compare `fault_candidates.gpkg` with the INGENIOUS favorability maps and known geothermal systems.
* Test the effect of the positive-unlabeled settings (`NEG_MIN_DIST_M`, `UNLABELED_WEIGHT`) and the block size on the fold scores.
* Remove one feature group at a time (lidar, magnetics, GNSS, seismicity) and compare the fold scores. This shows which data sets carry the signal.